# U2_T3A · Forma anidada de Horner, derivadas y deflación

### Evaluación eficiente de polinomios

## ¿Qué vas a aprender?

Evaluar un polinomio «como está escrito» es un desperdicio: calcular
$x^{4}$, $x^{3}$, $x^{2}$ por separado repite trabajo y **arrastra el error de redondeo de cada potencia**. La **regla de Horner** reescribe el polinomio de forma
anidada y consigue el mismo resultado con $n$ multiplicaciones en lugar de
$O(n^2)$.

De paso, esa misma pasada te regala **tres cosas más**: la derivada $P'(x_0)$
(con una segunda ejecución) y el **polinomio deflactado** $Q(x)$ de dividir entre
$(x-x_0)$.

| # | Pregunta | Tipo | Puntos |
|---|---|---|---|
| 1 | Los flotantes: $\varepsilon$, $\kappa$ y el error | opción múltiple | 10 |
| 2 | Cuánto ahorra la forma anidada | opción múltiple | 10 |
| 3 | Programar `horner(a, x0)` | código (casos ocultos) | 25 |
| 4 | Valor $P(x_0)$ de tu ejercicio | numérica | 10 |
| 5 | Derivada $P'(x_0)$ de tu ejercicio | numérica | 10 |
| 6 | Deflación: coeficientes de $Q(x)$ | 3 casillas | 25 |
| 7 | Decisión de ingeniería | opción múltiple | 10 |

$$\text{Calificación} = 0.70\,(\text{automático}) + 0.30\,(\text{2 actividades a mano})$$

Se requiere **$\ge 90\%$** de los 100 puntos automáticos para poder enviar.


In [1]:
#@title ⚙️ Configuración general — ejecuta esta celda **PRIMERO** (no modificar)
# ============================================================
# CELDA 1 — CONFIGURACIÓN GENERAL
# Detecta si corre en Google Colab o en tu computadora, monta Drive y
# carga el motor de la tarea (ver MODO_MOTOR más abajo).
# ============================================================
import base64
import importlib
import os
import subprocess
import sys
import zlib

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4.5)

# ¿Estamos en Google Colab?
try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

# --- Repositorio de la tarea  <-- EDITAR SOLO SI CAMBIA EL REPO ------
REPO_URL = "https://github.com/ITH-MGL-MN/U2_T3A.git"
REPO_NOMBRE = "U2_T3A"

# --- Solo en tu computadora: de dónde sale el motor ---
#   "modulos" -> profe/** tal cual, importado como paquete. Es el modo para
#                DEPURAR: Pylance lo entiende, así que funcionan F12 (ir a
#                definición), el autocompletado, el hover y los puntos de
#                ruptura, y estos caen en el código de verdad.
#   "bundle"  -> grader_local.py, el motor APLANADO en un solo archivo (igual
#                que el blob): para probar la forma exacta que corre en Colab.
#   "blob"    -> grader_ofuscado.txt, lo que recibe el alumno.
# En Colab siempre se usa el blob, haga lo que haga esta variable.
MODO_MOTOR = "modulos"

if ES_COLAB:
    BASE = "/content/drive/MyDrive/CursoMN"
    REPO = os.path.join(BASE, REPO_NOMBRE)
else:
    # En local: buscar la carpeta de la tarea (la que tiene grader_local.py)
    # subiendo de carpeta en carpeta desde la carpeta actual.
    REPO = os.getcwd()
    for _ in range(4):
        if os.path.exists(os.path.join(REPO, "grader_local.py")):
            break
        _padre = os.path.dirname(REPO)
        if _padre == REPO:
            break
        REPO = _padre
    # Respaldo: ruta local de la tarea en tu Drive (ajústala si la moviste)
    if not os.path.exists(os.path.join(REPO, "grader_local.py")):
        REPO = r"o:\Mi unidad\CursoMN\U2_T3A"

MODO = "blob" if ES_COLAB else MODO_MOTOR

print("Modo          :", "Google Colab" if ES_COLAB else "PC local")
print("Carpeta tarea :", REPO)

# ============================================================
# Montar Drive y traer el repositorio  (solo en Colab)
# En tu computadora no hace nada.
# ============================================================
if ES_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(BASE, exist_ok=True)
    try:
        if not os.path.exists(REPO):
            subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
        else:
            subprocess.run(["git", "-C", REPO, "pull"], check=True)
        print("✅ Repositorio listo en Drive.")
    except Exception as _exc:
        print("⚠️ No se pudo clonar/actualizar el repositorio:", _exc)
        print("   Si la carpeta ya existe en tu Drive puedes continuar.")
else:
    print("Modo local: no se monta Drive, uso la carpeta de arriba.")

# ============================================================
# Librerías necesarias
# ============================================================
for _pkg in ("sympy", "numpy", "scipy", "matplotlib", "requests", "ipywidgets"):
    try:
        importlib.import_module(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

# ============================================================
# Motor de la tarea
# ============================================================
# El motor vive aparte del cuaderno y este recibe solo la API que el motor
# declara en su `__all__`. Así el preludio de MATLAB de más abajo no puede
# pisarle los builtins (`max`, `sum`, `abs`, ...); ni el alumno, desde una celda
# de práctica, pisarle un nombre suyo. El motor sigue viendo el cuaderno por la
# pila de llamadas, así que `MI_TAREA` y los widgets `resp_i` viven aquí.
sys.path.insert(0, REPO)

if MODO == "modulos":
    # Import ESTÁTICO a propósito: es lo único que Pylance puede analizar, y el
    # `import *` respeta el `__all__` del motor (si el motor estrena un nombre,
    # el cuaderno lo ve sin tocar nada).
    from profe.ui.cuaderno import *      # noqa: F401,F403
else:
    if MODO == "blob":
        _ruta = os.path.join(REPO, "grader_ofuscado.txt")
        with open(_ruta, encoding="utf-8") as f:
            _blob = f.read().strip()
        codigo = zlib.decompress(base64.b64decode(_blob)).decode("utf-8")
    else:
        _ruta = os.path.join(REPO, "grader_local.py")
        with open(_ruta, encoding="utf-8") as f:
            codigo = f.read()

    _MOTOR = {}
    try:
        exec(codigo, _MOTOR)
    except Exception as _exc:
        raise RuntimeError(
            "No pude cargar el motor desde %s.\n"
            "En Colab: revisa que el repositorio se haya clonado.\n"
            "En local: ejecuta  python tools/build.py  para generarlo.\n"
            "Detalle: %r" % (_ruta, _exc)
        )
    globals().update({_n: _MOTOR[_n] for _n in _MOTOR["__all__"]})

print("Motor cargado :", MODO)

# ============================================================
# Preludio estilo MATLAB (el mismo de las demás tareas)
# ============================================================
# Deja a mano `linspace`, `zeros`, `ones`, `eye`, `pi`, `sin`, `plot`, ... y las
# funciones de NumPy también SIN el prefijo `np.`. Ojo: eso incluye `max`, `min`,
# `sum`, `abs`, `any`, `all` y `round`, que pasan a ser las de NumPy (por eso el
# cuaderno avisa de `max(a, b)` más abajo).
sys.path.insert(0, os.path.join(REPO, "lib"))
with open(os.path.join(REPO, "lib", "matlab_like.py"), encoding="utf-8") as f:
    exec(f.read(), globals())

# El preludio trae su propio tamaño de figura; el del cuaderno manda.
plt.rcParams["figure.figsize"] = (9, 4.5)
print("Preludio MATLAB:", os.path.join(REPO, "lib", "matlab_like.py"))

# ============================================================
# Herramientas de visualización (tablas y gráficas de los laboratorios)
# ============================================================
# Viven en tools/visualizar.py, FUERA del motor: no forman parte del sistema de
# calificación, así que no hay nada que esconder. Puedes leerlas, copiarlas y
# usarlas en tus propios cuadernos.
sys.path.append(os.path.join(REPO, "tools"))
try:
    import visualizar
    # OJO: `import` no vuelve a leer el archivo si el módulo ya estaba cargado,
    # así que al editar tools/visualizar.py hay que recargarlo o esta celda
    # seguiría usando la copia vieja que quedó en memoria.
    importlib.reload(visualizar)
    from visualizar import (bits_flotante, graficar_errores, graficar_polinomio,
                            latex_cientifico, tabla_experimento)
except ImportError as _exc:
    raise RuntimeError(
        "No encontré tools/visualizar.py, que los laboratorios usan para\n"
        "imprimir sus tablas y gráficas. En la carpeta de la tarea ese archivo\n"
        "debe estar junto al cuaderno.\nDetalle: %r" % (_exc,)
    )
print("Visualización :", os.path.join(REPO, "tools", "visualizar.py"))

print("\nListo. Ahora ejecuta la celda de identificación.")

Modo          : PC local
Carpeta tarea : o:\Mi unidad\CursoMN\U2_T3A
Modo local: no se monta Drive, uso la carpeta de arriba.
🔧 Modo debug local: cargando el bundle legible (sin ofuscar)
Motor cargado : BUNDLE local (grader_local.py)
API del motor : 35 nombres
Preludio MATLAB: o:\Mi unidad\CursoMN\U2_T3A\lib\matlab_like.py
Visualización : o:\Mi unidad\CursoMN\U2_T3A\tools\visualizar.py

Listo. Ahora ejecuta la celda de identificación.


---
## Identifícate

Al ejecutar la celda de configuración, Google te pidió permiso para acceder a
tu Drive. Con esa **misma cuenta** te identifico automáticamente: no tienes que
escribir nada.

> # ⚠️ INICIA SESIÓN CON TU CORREO INSTITUCIONAL
> - Debe ser un correo que termine en **@….tecnm.mx** (por ejemplo `@hermosillo.tecnm.mx`).
> - Si inicias con una cuenta personal, verás una alerta y **no podrás continuar**.
> - Es importante para identificarte y validar tu tarea.

Si ya ejecutaste esta celda varias veces, puede seguirte preguntando por otros
permisos: puedes continuar sin seleccionarlos. Solo se necesita el permiso de
**leer tu correo**.

In [2]:
#@title 👤 Identificación automática
# ============================================================
# IDENTIFICACIÓN AUTOMÁTICA
# Usa el correo institucional (…@….tecnm.mx) con el que montaste Drive.
# ============================================================
if ES_COLAB:
    import requests
    from google.colab import auth
    import google.auth
    import google.auth.transport.requests

    # Autenticación explícita para asegurar el acceso a la identidad
    try:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        auth_request = google.auth.transport.requests.Request()
        creds.refresh(auth_request)
        info = requests.get(
            "https://www.googleapis.com/drive/v3/about?fields=user",
            headers={"Authorization": "Bearer " + creds.token},
            timeout=20,
        ).json()
        email = info.get("user", {}).get("emailAddress", "")
    except Exception as e:
        print("⚠️ No se pudo obtener tu correo automáticamente.")
        print("Asegúrate de permitir el acceso en la ventana emergente.")
        print("Detalle:", e)
        email = ""
else:
    # Modo local (pruebas): escribe un correo manualmente
    email = "m16330887@hermosillo.tecnm.mx"
    # email = input("Correo institucional (pruebas): ").strip()

# --- Validación del dominio institucional ----------------------------
if email and ("@" in email) and email.lower().endswith(".tecnm.mx"):
    alumno_id = email
    print("✅ Identificado:", email)
    print("   Número de control:", extraer_nc(email))
elif not email:
    alumno_id = None
    print("⛔ No pude identificar tu correo.")
    print("   Vuelve a ejecutar la celda de configuración y acepta el permiso")
    print("   en la ventana de Google.")
else:
    from IPython.display import HTML

    display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;border-radius:8px;'
        'font-weight:bold">⛔ Este correo no es institucional.<br>'
        'Usa tu cuenta @….tecnm.mx. Si iniciaste con otra cuenta, '
        'cierra sesión o usa una ventana de incógnito.</p>'
    ))
    print("Correo detectado:", email)
    alumno_id = None

if alumno_id is None:
    raise RuntimeError(
        "Detente aquí: necesito tu correo institucional para generar tu tarea.\n"
        "Corrige la identificación y vuelve a ejecutar esta celda."
    )

# La tarea se genera UNA sola vez, aquí: las celdas de pregunta solo llaman a
# `pregunta(n)` sobre este MI_TAREA, así que todas hablan de la misma tarea.
MI_TAREA = generar_tarea(alumno_id)

print("\nListo. Continúa con la sección 1.")

✅ Identificado: m16330887@hermosillo.tecnm.mx
   Número de control: 16330887


Esta tarea tiene **7** preguntas automáticas (100 puntos) y **2** actividades a mano que revisa tu profesor (30% de la calificación).


Listo. Continúa con la sección 1.


---
# 1 · El costo de evaluar un polinomio

Un polinomio de grado $n$ se escribe

$$P(x) = a_n x^n + a_{n-1} x^{n-1} + \dots + a_1 x + a_0$$

y lo natural es evaluarlo **tal como está escrito**: calcular cada potencia
$x^2, x^3, \dots, x^n$ y multiplicarla por su coeficiente. Si se hace así, contar
las multiplicaciones da

$$n + (n-1) + (n-2) + \dots + 1 = \frac{n(n+1)}{2} = O(n^2)$$

Para un polinomio de grado 4 son **10** multiplicaciones; para grado 8, **36**.
En un microcontrolador de 8 bits cada multiplicación es una operación costosa
(decenas de ciclos), así que esto se nota.

## La forma anidada

Sacando factor común $x$ una y otra vez, el mismo polinomio se reescribe

$$P(x) = \bigl(\dots\bigl((a_n x + a_{n-1})x + a_{n-2}\bigr)x + \dots\bigr)x + a_0$$

y ahora **cada paso reutiliza el resultado del anterior**. Eso baja el costo a
exactamente $n$ multiplicaciones y $n$ sumas: $O(n)$.

---

> **También hay un efecto numérico, pero no hay que exagerarlo.** La forma anidada
> **no calcula las potencias** $x^2, x^3, \dots, x^n$, así que no arrastra el error
> de esa cadena de multiplicaciones; a cambio, encadena sus $n$ operaciones, una
> sobre el resultado de la anterior. El resultado neto es que Horner **suele ser
> algo más preciso**, pero por factores pequeños y **sin garantía**: quien decide
> cuántos dígitos se pierden en una evaluación es su **condición** ($\kappa$), no
> el método. **En el segundo laboratorio lo vas a medir con números, no con
> palabras**: primero con un polinomio normal (donde los dos empatan) y después
> con uno mal condicionado (donde los dos fallan).

---

### 🧪 Laboratorio A · ¿Cuántas multiplicaciones ahorra Horner?

El laboratorio de abajo **cuenta las multiplicaciones reales** y compara el tiempo de ejecución entre la implementación directa y la anidada.

**Las dos se miden en igualdad de condiciones:** Python puro, escalar, el mismo
número de repeticiones y sin bibliotecas compiladas. Comparar contra `numpy`
(que está escrita en C y procesa vectores completos) mediría la diferencia de
lenguajes, no la del algoritmo.

#### Glosario

> **Algoritmo «ingenuo» (*Naive algorithm*):** En computación y ciencias de la computación, el término *ingenuo* o del francés *naïve* se refiere al enfoque directo o literal de resolver un problema matemático tal como está definido en papel, sin aplicar optimizaciones algorítmicas ni considerar el consumo de recursos computacionales.

> **Regla de Horner:** Llamado así en honor al matemático británico William George Horner (aunque conocido siglos antes por matemáticos en China e India), es el algoritmo estándar para reducir la evaluación de un polinomio a una secuencia anidada de $n$ multiplicaciones y $n$ sumas.


In [3]:
#@title 🧪 Laboratorio A · Ahorro de operaciones y tiempo (mismas condiciones)
import time

filas = []
for n in (2, 4, 8, 12, 20, 50, 100, 200):
    a = [1.0] * (n + 1)
    x = 1.5

    v_ing, mult_ing = evaluar_ingenuo(a, x)     # valor y multiplicaciones contadas
    v_hor, mult_hor = evaluar_horner(a, x)      # el mismo trabajo: solo el valor
    v_motor = horner_referencia(a, x)[0]        # solo para comprobar el valor

    # Las mismas repeticiones para los dos, y suficientes para que el cronómetro
    # mida algo: al dividir entre ellas queda en microsegundos por evaluación.
    reps = np.maximum(5, 200000 // np.maximum(1, mult_ing))
    t0 = time.perf_counter()
    for _ in range(reps):
        evaluar_ingenuo(a, x)
    t1 = time.perf_counter()
    t2 = time.perf_counter()
    for _ in range(reps):
        evaluar_horner(a, x)
    t3 = time.perf_counter()

    def _cerca(u, v):
        return abs(u - v) <= 1e-12 * np.maximum(1.0, abs(u))

    iguales = "sí" if (_cerca(v_ing, v_motor) and _cerca(v_hor, v_motor)) else "⚠️ no"

    filas.append([n, mult_ing, mult_hor, mult_ing - mult_hor,
                  (t1 - t0) / reps * 1e6, (t3 - t2) / reps * 1e6, iguales])

tabla_experimento(
    [('Grado ($n$)', 'c', 'entero'),
     ('Mult. término a término', 'c', 'entero'),
     ('Mult. Horner', 'c', 'entero'),
     ('Ahorro', 'c', lambda v: '**$%d$**' % v),
     ('Tiempo ingenuo (µs)', 'c', '%.2f'),
     ('Tiempo Horner (µs)', 'c', '%.2f'),
     ('¿Mismo valor?', 'c')],
    filas,
    titulo='### 📊 Costo y tiempo (mismo lenguaje, mismas repeticiones)')


### 📊 Costo y tiempo (mismo lenguaje, mismas repeticiones)

| Grado ($n$) | Mult. término a término | Mult. Horner | Ahorro | Tiempo ingenuo (µs) | Tiempo Horner (µs) | ¿Mismo valor? |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| $2$ | $3$ | $2$ | **$1$** | 0.51 | 0.30 | sí |
| $4$ | $10$ | $4$ | **$6$** | 0.74 | 0.38 | sí |
| $8$ | $36$ | $8$ | **$28$** | 1.11 | 0.60 | sí |
| $12$ | $78$ | $12$ | **$66$** | 1.50 | 0.68 | sí |
| $20$ | $210$ | $20$ | **$190$** | 2.28 | 0.99 | sí |
| $50$ | $1275$ | $50$ | **$1225$** | 5.68 | 2.10 | sí |
| $100$ | $5050$ | $100$ | **$4950$** | 14.17 | 4.29 | sí |
| $200$ | $20100$ | $200$ | **$19900$** | 21.40 | 8.06 | sí |

> **¿Por qué el tiempo no baja tanto como las multiplicaciones?** Porque en Python
> `x ** k` es **una sola llamada a C** que hace toda la potencia por dentro: las
> 20 100 multiplicaciones «de papel» son en realidad ~200 llamadas de biblioteca.
> Por eso con $n=200$ Horner sale unas 2.6 veces más rápido y no 100 veces. En un
> microcontrolador (o en C, sin esa función) cada potencia **sí** son
> multiplicaciones de verdad y la diferencia se acerca a la de la tabla de
> operaciones: es justo el escenario que motiva el algoritmo.

**¿Qué se cronometra?** Solo el recorrido que calcula el valor, en los dos casos.
Ojo: el `horner` que tienes que programar hace eso *y además* construye el
cociente $Q$, así que su tiempo sería un poco mayor; aquí se compara la estrategia,
no el contrato de la función.

**¿Y numpy?** Aquí **no** se compara contra `numpy.polyval` a propósito: esa
función está escrita en C y procesa vectores completos, así que ganaría por
lenguaje, no por algoritmo. Dos implementaciones se comparan cambiando **una sola
cosa**: cómo se recorre el polinomio.

**¿Mismo valor?** Con coeficientes positivos los tres aciertan. La diferencia
numérica aparece cuando hay **cancelación**, y eso lo mide el laboratorio B.


#### 💡 ¿De dónde sale ese $2^{-52}$ y qué representa?

Un flotante funciona igual que la **notación científica**, pero en base 2:

$$\text{Número} = \pm\,(1.\text{mantisa}_2) \times 2^{\text{exponente}}$$

Como se reservan **52 bits para la mantisa** (los dígitos significativos), el
menor bit que la computadora puede sumar a la unidad es $2^{-52}$. Por eso,
**alrededor del $1.0$**, el siguiente número representable es $1.0 + 2^{-52}$, y
la distancia mínima en esa escala es exactamente
$\varepsilon \approx 2.22\times10^{-16}$.

Al alejarnos del $1.0$ esa distancia crece con la magnitud del número
($\Delta x \approx |x|\cdot\varepsilon$; el valor exacto queda entre
$\frac{|x|\varepsilon}{2}$ y $|x|\varepsilon$, así que la fórmula es una cota):

* **En escala astronómica ($x = 10^{15}$ m):** el salto al siguiente número en memoria es de **12.5 cm** (`np.spacing(1e15)` = 0.125). A esa escala no existen los centímetros: al ejecutar `1e15 + 0.01 == 1e15` (siendo `==` el operador de comparación de igualdad booleana) nos dará como resultado `True`.
* **En escala microscópica ($x = 10^{-12}$ m):** el salto es
  $\approx 2.22\times10^{-28}$ m, una resolución física diminuta, pero
  **manteniendo exactamente las mismas 16 cifras significativas en relativo**.

---

De ahí, dos consecuencias esenciales:

1. **El error siempre es relativo:** $10^{155}$ de error absoluto puede ser
   insignificante o catastrófico según la magnitud del resultado.
2. **Cancelación por resta:** al restar números muy cercanos de gran magnitud, las
   cifras significativas reales se desvanecen. Cuánto se pierde lo anticipa el
   **número de condición** ($\kappa$):

$$\frac{|\text{error}|}{|P(x)|} \approx \kappa\,\varepsilon,
\qquad \kappa = \frac{\sum_i |a_i\,x^{\,n-i}|}{|P(x)|}$$

En la **celda de bits** de abajo puedes cambiar la precisión y ver qué se mueve;
en los **dos laboratorios** que siguen, medir todo esto con tu polinomio:

* **B1 — bien condicionado:** un polinomio normal, donde $\kappa$ es pequeño y los
  dos métodos empatan.
* **B2 — mal condicionado:** $(x-1)^n$ desarrollado, donde la cancelación se lleva
  todos los dígitos y fallan los dos.

In [4]:
#@title 🔬 Cómo se guarda un número: sus bits
# Los bits del EXPONENTE dan el rango; los de la MANTISA, la precisión.
# Escribe otro número o cambia la precisión, y mira qué se mueve.
import ipywidgets as widgets

tipo = widgets.Dropdown(options=["float64", "float32"], description="Precisión:")
numero = widgets.FloatText(value=0.1, description="Número:")
vista = widgets.interactive_output(bits_flotante,
                                   {"numero": numero, "precision": tipo})
display(widgets.VBox([widgets.HBox([tipo, numero]), vista]))

In [5]:
#@title Laboratorio B1 · Los últimos dígitos, a la vista
# El épsilon lo da la biblioteca (2**-52 en doble precisión), no se escribe a mano.
EPS = np.finfo(float).eps

a = [2.0, -2.0, 7.0, -2.0, 7.0]
x = 1.7
exacto = evaluar_exacto(a, x)      # con fracciones: no redondea nada
v_ing, _ = evaluar_ingenuo(a, x)
v_hor = horner_referencia(a, x)[0]

# La fila de la referencia no tiene error que medir: se pasa None y la tabla
# dibuja un guion.
filas = []
for nombre, valor in (("exacto (fracciones)", float(exacto)),
                      ("término a término", v_ing),
                      ("Horner", v_hor)):
    filas.append([nombre, valor,
                  None if nombre.startswith("exacto")
                  else error_relativo(valor, exacto)])

tabla_experimento(
    [('Evaluación', 'l'),
     ('$P(1.7)$ (17 cifras)', 'c', '$%.17g$'),
     ('Error relativo', 'c', 'ciencia2')],
    filas)

tabla_experimento(
    [('Magnitud', 'l'), ('Valor', 'l')],
    [['Diferencia entre los dos métodos',
      '$%s$  (**%d ulp**)' % (latex_cientifico(abs(v_ing - v_hor)),
                              round(abs(v_ing - v_hor) / np.spacing(v_ing)))],
     ['$\\varepsilon$ de la máquina (`np.finfo(float).eps`)',
      '$%s$' % latex_cientifico(EPS, 16)],
     ['$\\kappa$ de esta evaluación', '$%.2f$' % condicion_evaluacion(a, x)]])


| Evaluación | $P(1.7)$ (17 cifras) | Error relativo |
|:---|:---:|:---:|
| exacto (fracciones) | $30.708199999999998$ | — |
| término a término | $30.708199999999998$ | $0$ |
| Horner | $30.708199999999994$ | $1.16\times 10^{-16}$ |

| Magnitud | Valor |
|:---|:---|
| Diferencia entre los dos métodos | $3.553\times 10^{-15}$  (**1 ulp**) |
| $\varepsilon$ de la máquina (`np.finfo(float).eps`) | $2.2204460492503131\times 10^{-16}$ |
| $\kappa$ de esta evaluación | $1.86$ |

El polinomio $a = [2, -2, 7, -2, 7]$ evaluado en $x = 1.7$ de tres formas: la
**exacta** (con fracciones, sin redondear nada) y las dos programadas. La columna
del error es **relativo**, así que dice directamente cuántos dígitos se perdieron
en cada método.

Las tres últimas filas de la segunda tabla son las unidades con que se juzga:

* **ulp** (*unit in the last place*): la distancia entre dos números
  representables consecutivos. Una diferencia de 1 ulp es lo mínimo que un `float`
  distingue, así que aquí los dos métodos están **pegados al límite**.
* **$\varepsilon$**: el épsilon de la máquina, el «peldaño» de la escalera de
  números (el $2^{-52}$ de arriba).
* **$\kappa$**: el número de condición de la evaluación. Con $\kappa$ pequeño el
  problema está **bien condicionado**, $\kappa\varepsilon$ queda muy por debajo de
  1 y **no hay dígitos que ganar**. Si Horner vale aquí, es por el **costo**.


In [6]:
#@title 🧪 Laboratorio B2 · Cuando la evaluación está mal condicionada
# (x - 1)^n DESARROLLADO en x = 1.0001: los términos llegan a 1e17 y el resultado
# es 1e-4^n, así que la resta se lleva los dígitos.
EPS = np.finfo(float).eps

filas = []
x2 = 1 + 1e-4
for n in (2, 6, 12, 20, 40):
    a2 = coeficientes_binomio(n)
    ex2 = evaluar_exacto(a2, x2)
    v1, _ = evaluar_ingenuo(a2, x2)
    v2 = horner_referencia(a2, x2)[0]
    k = condicion_evaluacion(a2, x2)
    filas.append([n, k, error_relativo(v1, ex2), error_relativo(v2, ex2), k * EPS])

tabla_experimento(
    [('$n$', 'c', 'entero'),
     ('$\\kappa$', 'c', 'ciencia'),
     ('Error término a término', 'c', 'ciencia2'),
     ('Error de Horner', 'c', 'ciencia2'),
     ('$\\kappa\\varepsilon$', 'c', 'ciencia2')],
    filas,
    titulo="Con $P(x) = (x-1)^n$ **desarrollado** en $x = 1.0001$: el valor "
           "exacto es $10^{-4n}$ y los términos del desarrollo llegan a "
           "$10^{17}$.\n\n### 📊 Error medido contra el valor exacto")


Con $P(x) = (x-1)^n$ **desarrollado** en $x = 1.0001$: el valor exacto es $10^{-4n}$ y los términos del desarrollo llegan a $10^{17}$.

### 📊 Error medido contra el valor exacto

| $n$ | $\kappa$ | Error término a término | Error de Horner | $\kappa\varepsilon$ |
|:---:|:---:|:---:|:---:|:---:|
| $2$ | $4.000\times 10^{8}$ | $6.08\times 10^{-9}$ | $5.02\times 10^{-9}$ | $8.88\times 10^{-8}$ |
| $6$ | $6.402\times 10^{25}$ | $3.55\times 10^{9}$ | $8.88\times 10^{8}$ | $1.42\times 10^{10}$ |
| $12$ | $4.098\times 10^{51}$ | $2.10\times 10^{35}$ | $8.62\times 10^{34}$ | $9.10\times 10^{35}$ |
| $20$ | $1.050\times 10^{86}$ | $2.04\times 10^{68}$ | $1.57\times 10^{67}$ | $2.33\times 10^{70}$ |
| $40$ | $1.102\times 10^{172}$ | $1.35\times 10^{155}$ | $1.49\times 10^{154}$ | $2.45\times 10^{156}$ |

El caso opuesto al anterior: $P(x) = (x-1)^n$ **desarrollado**, evaluado en
$x = 1.0001$.

* Los términos del desarrollo valen hasta $10^{17}$, pero el resultado verdadero
  es $(10^{-4})^n$, minúsculo. Al restarlos se **cancelan** y con ellos se van los
  dígitos: es la **cancelación por resta** que se describió arriba.
* Por eso $\kappa$ explota, y la última columna ($\kappa\varepsilon$) **predice**
  el error de las dos anteriores. Cuando $\kappa\varepsilon$ pasa de 1, no queda
  **ninguna** cifra buena: da igual qué método se use.


### Lo que dicen las dos tablas

* **B1**, bien condicionado ($\kappa \approx 2$): los dos métodos coinciden hasta
  el **último** dígito binario (1 ulp). Con $\kappa$ pequeño no hay dígitos que ganar:
  Horner vale por el **costo**.
* **B2**, mal condicionado ($\kappa$ enorme): los dos pierden **todos** los
  dígitos, y por poco parecido. El error sigue a $\kappa\varepsilon$, que ya pasó
  de 1: no queda ninguna cifra buena que rescatar.
* Con $\kappa \approx 1$ el error es del orden de $\varepsilon$, y **ningún**
  método baja de ahí.
* Moraleja: quien decide cuántos dígitos se conservan es la **condición del
  problema**, no el método. Lo que Horner garantiza siempre es el costo.


In [7]:
#@title 🧪 Pregunta 1 · los flotantes: $\varepsilon$, $\kappa$ y el error
pregunta(1)

#### Pregunta 1: Qué cambia al pasar a float32

**Valor:** $\frac{10}{100}$ puntos

En la celda de bits se ve que `0.1` en `float32` se guarda como `0.100000001490116119384765625` y en `float64` como `0.1000000000000000055511151231257827021181583404541015625`. **¿Qué cambia entre las dos precisiones?**

a) Solo el rango: se guardan las mismas 16 cifras, pero con exponentes más pequeños

b) Solo el número de cifras que se imprimen; el valor guardado es el mismo

c) Nada: Python convierte siempre a `float64` antes de guardar

d) Las dos cosas: el rango (8 bits de exponente en lugar de 11) y la precisión (23 bits de mantisa en lugar de 52)

RadioButtons(layout=Layout(width='170px'), options=(('a)', 0), ('b)', 1), ('c)', 2), ('d)', 3)), value=None)

### 1. La Recurrencia (Evaluación en una sola pasada)

Cuando evaluamos un polinomio $P(x) = a_n x^n + a_{n-1} x^{n-1} + \dots + a_0$ de la forma tradicional, calcular las potencias $x^2, x^3, \dots, x^n$ resulta costoso y propensó a acumular errores de redondeo.

Horner reescribe el polinomio en su **forma anidada**:


$$P(x) = \Big(\big(a_n x + a_{n-1}\big)x + a_{n-2}\Big)x + \dots + a_0$$

Para programarlo o hacerlo a mano, esto se traduce en una **cadena de acumulación** que recorre los coeficientes de **mayor a menor grado** ($a_n$ hasta $a_0$):

* **Paso inicial:** $b_n = a_n$

* **En el bucle:** $b_k = a_k + b_{k+1}\,x_0$

#### ¿Qué te regala la primera ejecución?

1. **El valor del polinomio:** El último número que calculas ($b_0$) es exactamente el valor de la función en ese punto: **$P(x_0) = b_0$**.


2. **El polinomio cociente $Q(x)$:** Todos los números anteriores ($b_n, b_{n-1}, \dots, b_1$) son los **coeficientes del cociente** de dividir $P(x)$ entre $(x - x_0)$.



> **Cuidado con el orden:** Si tu polinomio original es de grado $n$, la lista de coeficientes de entrada $a$ tiene tamaño $n+1$. Los valores $b_n \dots b_1$ forman un nuevo polinomio $Q(x)$ de grado $n-1$.
> 

### 2. La Derivada (Segunda ejecución sin derivar a mano)

Normalmente, para evaluar la derivada tendría que calcularse $P'(x)$ aplicando reglas de derivación término a término. Horner evita esto reutilizando la misma lógica.

Si se aplica la misma recurrencia **sobre los valores $b_k$** que se obtuvieron en la primera pasada:

* **Paso inicial:** $c_n = b_n$

* **En el bucle:** $c_k = b_k + c_{k+1}\,x_0$


Al terminar esta segunda ejecución, **el penúltimo valor** ($c_1$) es la **primera derivada**:


$$\boxed{P'(x_0) = c_1}$$

> **¿Por qué $c_1$ y no $c_0$?**
> Al dividir $P(x)$ entre $(x - x_0)$ se obtiene $$P(x) = (x - x_0)Q(x) + b_0.$$ Si se deriva esta expresión usando la regla del producto y se evalúa en $x_0$, el resultado muestra analíticamente que $P'(x_0) = Q(x_0)$. Como la segunda fila calcula precisamente $Q(x_0)$, el valor de la función $Q$ en $x_0$ queda ubicado en el penúltimo término ($c_1$).
> 

### 3. La Deflación (Reducción de grado)

Cuando se utiliza un método numérico (como Newton-Raphson) para encontrar una raíz $r$ de un polinomio de grado $n$, volver a buscar más raíces sobre el mismo polinomio puede hacer que el método converja **otra vez a la misma raíz**.

Para evitarlo se utiliza la **deflación**:

1. Si $r$ es una raíz exacta, entonces $P(r) = 0$, lo que implica que el residuo de la primera ejecución es cero ($b_0 = 0$).

2. La relación de la división queda simplemente como:

$$P(x) = (x - r) \, Q(x)$$

3. Ahora solo se deben buscar las raíces restantes en **$Q(x)$**, que tiene **un grado menos $(n-1)$**.

> **PUNTOS IMPORTANTES**
> * **Conserva el coeficiente líder:** Si el polinomio original empieza con $4x^3$, el polinomio reducido $Q(x)$ empezará con $4x^2$. El primer coeficiente siempre se pasa directo ($b_n = a_n$).
> 
> 
> * **¿Qué pasa si el residuo $b_0 \neq 0$?** Significa que $r$ no era una raíz exacta, sino una **aproximación**. El valor de $b_0$ representa justamente el residuo o error $P(r)$.
> 
> 
> 
>

In [8]:
#@title 🔬 Laboratorio · la división sintética paso a paso
# Un polinomio de ejemplo cualquiera (NO es el tuyo).
a_demo = [2, -6, 2, -1]
x_demo = 3

In [9]:
import sympy as sp
from IPython.display import Math, Markdown, display


def _exacto(v):
    '''Convierte a número EXACTO de sympy: el decimal que escribiste, no el
    binario que en realidad se guarda (0.1 -> 1/10, exacto).'''
    if isinstance(v, sp.Basic):
        return v
    if isinstance(v, bool):          # bool es subclase de int: va antes
        return sp.Integer(int(v))
    if isinstance(v, int):
        return sp.Integer(v)
    if isinstance(v, float):
        return sp.Rational(str(v))
    return sp.Rational(v)


def _tex(v):
    '''Un número exacto listo para meter en LaTeX.'''
    return sp.latex(_exacto(v))


def division_sintetica_detallada(a, x0):
    '''Muestra los dos recorridos de la división sintética, paso a paso.

    Los números se calculan como fracciones exactas de sympy, así que nada de
    lo que ves en pantalla lleva redondeo. Solo dibuja: no devuelve nada.
    '''
    a = [_exacto(c) for c in a]
    x0 = _exacto(x0)
    n = len(a) - 1
    x = sp.Symbol('x')

    display(Markdown("**Coeficientes** $a = %s$   y   $x_0 = %s$"
                     % (sp.latex(a), sp.latex(x0))))

    # ------------------- Primera ejecución: los b -------------------
    display(Markdown("#### Primera ejecución ($b$)"))
    b = [a[0]]
    display(Math(r"b_{%d} = a_{%d} = %s" % (n, n, _tex(b[0]))))
    for i in range(1, n + 1):
        k = n - i
        b.append(a[i] + b[i - 1] * x0)
        display(Math(r"b_{%d} = a_{%d} + b_{%d}\,x_0 = %s + \left(%s\right)"
                     r"\left(%s\right) = \mathbf{%s}"
                     % (k, k, k + 1, _tex(a[i]), _tex(b[i - 1]), _tex(x0),
                        _tex(b[-1]))))
    display(Math(r"\boxed{\ P(x_0) = b_0 = %s\ }" % _tex(b[-1])))
    Q = sp.Poly(b[:-1], x).as_expr() if len(b) > 1 else sp.Integer(0)
    display(Math(r"Q(x) = %s" % sp.latex(Q)))

    # ------------------- Segunda ejecución: los c -------------------
    display(Markdown("#### Segunda ejecución ($c$), sobre los $b$"))
    if n == 0:
        # Un polinomio constante no cambia: no hay segunda ejecución que hacer.
        display(Math(r"\boxed{\ P'(x_0) = 0\ }"))
    else:
        c = [b[0]]
        for i in range(1, n):
            k = n - i
            c.append(b[i] + c[i - 1] * x0)
            display(Math(r"c_{%d} = b_{%d} + c_{%d}\,x_0 = %s + \left(%s\right)"
                         r"\left(%s\right) = \mathbf{%s}"
                         % (k, k, k + 1, _tex(b[i]), _tex(c[i - 1]), _tex(x0),
                            _tex(c[-1]))))
        display(Math(r"\boxed{\ P'(x_0) = c_1 = %s\ }" % _tex(c[-1])))

    # ------------------- Comprobación: P(x) = (x-x0)Q(x) + b0 -------------------
    P = sp.expand(sp.Add(*[a[i] * x ** (n - i) for i in range(n + 1)]))
    derecha = sp.expand((x - x0) * Q + b[-1])
    display(Markdown("#### Comprobación de la división"))
    display(Math(r"P(x) = %s \qquad (x - x_0)\,Q(x) + b_0 = %s"
                 % (sp.latex(P), sp.latex(derecha))))
    display(Math(r"P(x) - \left[(x - x_0)\,Q(x) + b_0\right] = %s"
                 % sp.latex(sp.simplify(P - derecha))))

    display(Markdown("La gracia es que **no aparece ninguna potencia de $x$ "
                     "elevada**: cada $b$ se calcula con el $b$ anterior."))


division_sintetica_detallada(a_demo, x_demo)

**Coeficientes** $a = \left[ 2, \  -6, \  2, \  -1\right]$   y   $x_0 = 3$

#### Primera ejecución ($b$)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

#### Comprobación de la división

<IPython.core.display.Math object>

<IPython.core.display.Math object>

La gracia es que **no aparece ninguna potencia de $x$ elevada**: cada $b$ se calcula con el $b$ anterior.

---

### ✏️ Pregunta 2


In [10]:
#@title 🧪 Pregunta 2 · cuánto ahorra la forma anidada
pregunta(2)

#### Pregunta 2: Para qué sirve deflactar

**Valor:** $\frac{10}{100}$ puntos

Ya conoces una raíz exacta $r$ de $P(x)$ (es decir $P(r)=0$). Quieres encontrar **las demás** raíces. **¿Qué ganas al deflactar?**

a) Nada: la raíz $r$ sigue apareciendo en el polinomio reducido

b) Que la raíz $r$ se vuelva doble

c) Bajas el grado en 1, así que el siguiente método trabaja con un problema más simple

d) Convierte el polinomio en uno de primer grado siempre

RadioButtons(layout=Layout(width='170px'), options=(('a)', 0), ('b)', 1), ('c)', 2), ('d)', 3)), value=None)

---
# 3 · Actividad a mano · Horner: $P(x_0)$ y $P'(x_0)$

Aquí abajo está **tu** polinomio (el tuyo, no el de tu compañero) y el punto de
evaluación. Vas a llenar la tabla **a mano, con calculadora**, en este orden:

1. **Fila $b_k$** (primera ejecución): empieza con $b = a_n$ y aplica
   $b \leftarrow b \cdot x_0 + a_k$ recorriendo los coeficientes de mayor a menor
   grado. El **último** valor que obtengas es $P(x_0)$.
2. **Fila $c_k$** (segunda ejecución): repite lo mismo **pero sobre los $b$** (no
   sobre los $a$). El **penúltimo** valor de esta fila es $P'(x_0)$.

Después reporta tus dos resultados en la celda que sigue. **No redondees los
pasos intermedios**: el error se acumula y la comprobación te lo va a decir.

> ⚠️ **Ojo al escribir tu código.** La configuración carga `matlab_like` con
> sintaxis estilo MATLAB (`from numpy import *`), así que `max`, `min`, `abs`,
> `any`, `all`, `sum` y `round` son las de **NumPy**, no las de Python. En
> concreto, `max(a, b)` **falla** (`AxisError`: NumPy lee el segundo valor como un
> eje). Para comparar **dos valores**:
>
> ```python
> np.maximum(a, b)   # en vez de max(a, b)
> np.minimum(a, b)   # en vez de min(a, b)
> ```
>
> A cambio tienes a mano, sin prefijo, las funciones de MATLAB: `linspace`,
> `zeros`, `ones`, `eye`, `size`, `pi`, `sin`, `cos`, `sqrt`, `sum`, `plot`, ...


In [11]:
#@title ✏️✅ Horner · actividad a mano, práctica y preguntas
EJ_HR = mano_enunciado('HORNER')

### ✏️ Actividad a mano: Forma anidada de Horner

**Curva de un motor de CD (grado 3)**

El **par de un motor de CD** controlado por un variador sigue una curva de
tercer grado respecto a la corriente de armadura:

$$P(x) = x^{3} - 4x^{2} + 7x + 1$$

con $x$ en **amperes** y $P(x)$ en **N·m**.

El fabricante te pide el par y la pendiente de la curva (su derivada, que es
la constante de torque local) en $x_0 = 2.3$ A. Evalúa las dos con la
**forma anidada de Horner** y anota los pasos en la tabla: primero todos los
$b_k$, después todos los $c_k$.

**Datos y arranque:** $x_0 = 2.3$  ·  coeficientes `a = [1, -4, 7, 1]`

**Tu tabla para llenar a mano** (no redondees los pasos intermedios):

grado          a_k            b_k            c_k          
------------------------------------------------------------
3              1                                          
2              -4                                         
1              7                                          
0              1                                          


`grado` = la potencia que acompaña al coeficiente (de mayor a menor)  ·  `a_k` = los coeficientes del polinomio, que ya te da el enunciado  ·  `b_k` = la **primera corrida**: cada $b$ se calcula con el $b$ anterior  ·  `c_k` = la **segunda corrida**, que va sobre los $b$ (no sobre los $a$).

Reporta en la celda siguiente la lista `[P(x_0), P'(x_0)]`, con al menos 4 decimales. **Se comprueban todos los valores** para que verifiques tu tabla.

In [23]:
# ---------------------------------------------------------------------
#  TU ECUACION, escrita por ti como funciones lambda.
#  P  -> el polinomio del enunciado
#  dP -> su derivada analitica (la que obtendrias derivando a mano)
#
#  Por ejemplo:  P  = lambda x: x**3 - 4*x**2 + 7*x + 1
#                dP = lambda x: 3*x**2 - 8*x + 7
# ---------------------------------------------------------------------
P = None          # <-- escribe TU polinomio   (P = lambda x: ...)
dP = None         # <-- escribe TU derivada    (dP = lambda x: ...)

mano_ecuacion('HORNER', f=P, df=dP)

Revisión de TU ecuación (en 3 valores que no ves):
   P    : todavía no la escribiste
   P'   : todavía no la escribiste
⚠️ Falta escribir: P, P'


In [24]:
# ---------------------------------------------------------------------
#  Escribe aqui tus DOS valores calculados a mano y vuelve a ejecutar
#  esta celda para recibir la realimentacion.
# ---------------------------------------------------------------------
MIS_HR = [0.0, 0.0]          # <-- [P(x_0), P'(x_0)]

mano_comprueba('HORNER', MIS_HR)

Tus valores reportados: P(x_0) = 0, P'(x_0) = 0

   valor      tu resultado     referencia       estado
   ----------------------------------------------------------
   P(x_0)     0                8.107            ❌ (error 1.0e+00)
   P'(x_0)    0                4.47             ❌ (error 1.0e+00)

   Sugerencias: arrastra TODOS los decimales del paso anterior y no
   redondees a la mitad del cálculo.
   En la derivada, recuerda que la segunda corrida va sobre los
   b_k, y que P'(x_0) es el PENÚLTIMO valor de esa segunda fila.


---
### 💻 Pregunta 3 · programa el algoritmo

Ahora vas a programar exactamente lo que acabas de hacer a mano, pero para
cualquier polinomio. La pregunta pide una función con esta firma:

```python
horner(a, x0)  ->  (p_val, dp_val, Q)
```

* `a` es la lista de coeficientes **de mayor a menor grado**;
* devuelve el valor, la derivada y la **lista** del cociente.

Se califica con **casos ocultos** (polinomios que nunca ves), así que no sirve
devolver un número fijo ni llamar a `numpy.polyval`: hay que recorrer los
coeficientes con la recurrencia.

> Recuerda el aviso de la sección 3: con el preludio de MATLAB, `max(a, b)`
> falla; usa `np.maximum(a, b)`.


In [ ]:
#@title Pregunta 3 · programar horner con casos ocultos
pregunta(3)

#### Pregunta 2: Programa: Forma anidada de Horner

**Valor:** $\frac{25}{100}$ puntos

Programa el **algoritmo anidado de Horner** para evaluar un polinomio y su
primera derivada en un punto, usando **una sola pasada** por los coeficientes
para el valor y una **segunda pasada** (sobre los resultados de la primera)
para la derivada.

Tu función recibe:

* `a` — la lista de coeficientes del polinomio, **de mayor a menor grado**:
  $a=[a_n, a_{n-1}, \dots, a_0]$ (el polinomio tiene grado $n$)
* `x0` — el punto donde se evalúa

y devuelve **una tupla con tres elementos**, en este orden:

1. `p_val` — el valor $P(x_0)$
2. `dp_val` — la primera derivada $P\,'(x_0)$
3. `Q` — la **lista** de coeficientes del cociente de dividir $P(x)$ entre
   $(x-x_0)$, también de mayor a menor grado (son los $b$ de la primera
   pasada, sin el último)

Con el ejemplo `horner([1, -4, 5, -3, 2], 3.0)` debe devolver
`(11.0, 27.0, [1.0, -1.0, 2.0, 3.0])`: el polinomio evaluado en 3, su
derivada en 3 y el cúbico que resulta de dividirlo entre $(x-3)$.

**No uses `numpy.polyval` ni símilares**: se califica que tu código haga el
recorrido anidado. Fíjate en la diferencia con evaluar término a término: en
la forma anidada **cada resultado se reutiliza** como el factor de la
siguiente multiplicación.

Define la función con **exactamente** la firma `horner(a, x0)` en una celda aparte y ejecútala (no la definas dentro de esta celda).

In [26]:
# --- Práctica: programa el método ------------------------------------
def horner(a, x0):
    '''
    Forma anidada de Horner.

    Parametros
    ----------
    a  : lista de coeficientes de MAYOR a menor grado, [a_n, ..., a_0]
    x0 : punto de evaluacion

    Devuelve
    --------
    (p_val, dp_val, Q)
        p_val  : P(x0)
        dp_val : P'(x0)
        Q      : lista de los coeficientes del cociente P(x)/(x - x0),
                 tambien de mayor a menor grado
    '''
    # =================================================================
    #  ESCRIBE TU CODIGO AQUI
    #  Primera corrida:  b = a[0]; para cada coeficiente siguiente
    #      b = b*x0 + a_k        (ve guardando cada b)
    #  Segunda corrida:  igual, pero sobre los b_k ya guardados
    #      (el resultado P'(x0) es el PENULTIMO de esa fila)
    #  Y ojo: Q son los b SIN el ultimo.
    # =================================================================
    # return (p_val, dp_val, Q)
    raise NotImplementedError("Completa la funcion horner") # Borra esta línea


# --- Prueba con TU ejercicio -----------------------------------------
try:
    _p, _d, _q = horner(EJ_HR['a'], EJ_HR['x0'])
    comparar_practica('HORNER', EJ_HR, (_p, _d, _q))
except NotImplementedError as exc:
    print("Todavia no terminas la funcion:", exc)
except TypeError as exc:
    print("Tu funcion devolvio algo que no se puede leer:", exc)

# --- Solución de referencia (revisa DESPUES de intentarlo) -----------
tabla('HORNER', EJ_HR)

Todavia no terminas la funcion: Completa la funcion horner
Tabla de la división sintética (referencia)
grado          a_k            b_k            c_k          
------------------------------------------------------------
3              1              1              1            
2              -4             -1.7           0.6          
1              7              3.09           4.47         
0              1              8.107                       


---
### ✏️ Preguntas 4 y 5 · los dos valores de tu ejercicio

Ya tienes la tabla llena a mano y la función programada. Contesta con los
números de **tu** ejercicio (los mismos que reportaste en `MIS_HR`). Si quieres
comprobar antes, ejecuta `mano_solucion('HORNER')`.


In [ ]:
pregunta(4)      # el valor P(x_0)

#### Pregunta 3: P(x_0): Forma anidada de Horner

**Valor:** $\frac{15}{100}$ puntos

**Ejercicio:** Curva de un motor de CD (grado 3)

Con el **polinomio de tu ejercicio** (el mismo de la actividad a mano de esta
sección), llena la tabla de la división sintética y reporta **solo el valor**
$P(x_0)$.

El punto de evaluación $x_0$ es el que aparece en el enunciado de tu
actividad; no lo cambies.

Reporta un **número**, con al menos 4 decimales y **sin unidades**. Si ya
programaste `horner(a, x0)`, puedes comprobar tu resultado con ella: es el
primer valor que devuelve.

FloatText(value=0.0, description='Respuesta:', layout=Layout(width='260px'))

In [ ]:
pregunta(5)      # la derivada P'(x_0)

#### Pregunta 4: P'(x_0): Forma anidada de Horner

**Valor:** $\frac{15}{100}$ puntos

**Ejercicio:** Curva de un motor de CD (grado 3)

Con el **mismo polinomio de tu ejercicio**, reporta ahora **solo el valor de
la primera derivada** $P\,'(x_0)$ en ese punto.

Para obtenerla necesitas la **segunda corrida** de la división sintética:
vuelve a aplicar el mismo algoritmo, pero ahora sobre los coeficientes $b_k$
que obtuviste en la primera pasada. El resultado es $P\,'(x_0)=c_1$, es decir
el **penúltimo** valor de esa segunda fila.

Reporta un **número**, con al menos 4 decimales y **sin unidades**. Es el
segundo valor que devuelve tu función `horner(a, x0)`, así que puedes
comprobarlo con ella.

FloatText(value=0.0, description='Respuesta:', layout=Layout(width='260px'))

---
# 4 · Deflación: quitar una raíz que ya conoces

Cuando ya sabes que $r$ es raíz de $P$, dividir entre $(x-r)$ te deja un
polinomio de un grado menos **con las mismas raíces restantes**:

$$P(x) = (x - r)\,Q(x) + \underbrace{P(r)}_{b_0}$$

Esa división es **exactamente la primera corrida de Horner evaluada en $r$**, así
que no hay que programar nada nuevo: los coeficientes de $Q$ son los $b_k$ que
obtuviste, sin el último.

## ¿Para qué sirve?

Encontrar todas las raíces de un polinomio de grado alto es un problema difícil.
La estrategia estándar es:

1. encuentra **una** raíz (con Newton, Bairstow, Müller… o «por inspección»);
2. **deflacta** para bajar el grado;
3. repite con el polinomio reducido.

Y si el residuo no sale cero, es que la raíz era solo **aproximada**: ese residuo
es $P(r)$, el error que cometerías al dar por buena esa raíz.

> ⚠️ **El error clásico al deflactar a mano:** dividir como si el polinomio fuera
> mónico. Si $P$ empieza con $4x^3$, entonces $Q$ empieza con $4x^2$: el
> coeficiente líder **se conserva**.

In [29]:
#@title ✏️✅ Deflación · actividad a mano y pregunta
EJ_DF = mano_enunciado('DEFLACION')

### ✏️ Actividad a mano: Deflación polinomial

**Perfil de leva: deflación con una raíz exacta**

El perfil de una **leva mecánica** se modela con el polinomio cúbico

$$P(x) = x^{3} - 3x^{2} - 3x + 10$$

donde $x$ es el ángulo del árbol de levas.

Al graficarlo observas que la curva cruza el eje en $x = 2$, o sea que
$P(2) = 0$: es una **raíz exacta** y la conoces de antemano. No tiene
sentido volver a resolver el problema completo desde el principio.

Aplica **una división sintética** (deflación) dividiendo $P(x)$ entre
$(x - 2)$ y reporta los coeficientes del polinomio reducido de segundo
grado

$$Q(x) = A\,x^{2} + B\,x + C$$

**de mayor a menor grado**, es decir reporta $A$, luego $B$ y luego $C$.

La tabla de la división está en blanco abajo. Fíjate en que el último valor
que obtengas (el residuo) también es información: si $x = 2$ fuera raíz
solo *aproximada*, ese residuo sería el error que cometes al suponerla.

**Datos y arranque:** $r = 2$  ·  coeficientes `a = [1, -3, -3, 10]`

**Tu tabla para llenar a mano** (no redondees los pasos intermedios):

grado          a_k            b_k          
---------------------------------------------
3              1                           
2              -3                          
1              -3                          
0              10                          


`grado` = la potencia del coeficiente  ·  `a_k` = los coeficientes del polinomio original  ·  `b_k` = el cociente de dividir entre $(x-r)$; el **último** valor que obtengas es el residuo.

Reporta en la celda siguiente la lista `[A, B, C]`, con al menos 4 decimales. **Se comprueban todos los valores** para que verifiques tu tabla.

In [30]:
# ---------------------------------------------------------------------
#  TU ECUACION, escrita por ti como funcion lambda.
#  Es el MISMO polinomio del enunciado (el de arriba).
#
#  Por ejemplo:  P = lambda x: x**3 - 6*x**2 + 7*x + 6
# ---------------------------------------------------------------------
P = None          # <-- escribe TU polinomio   (P = lambda x: ...)

mano_ecuacion('DEFLACION', f=P)

Revisión de TU ecuación (en 3 valores que no ves):
   P    : todavía no la escribiste
⚠️ Falta escribir: P


In [31]:
# ---------------------------------------------------------------------
#  Escribe aqui los TRES coeficientes del polinomio reducido Q(x),
#  de MAYOR a menor grado, y vuelve a ejecutar esta celda.
# ---------------------------------------------------------------------
MIS_DF = [0.0, 0.0, 0.0]          # <-- [A, B, C]

mano_comprueba('DEFLACION', MIS_DF)

Tus valores reportados: A = 0, B = 0, C = 0

   valor      tu resultado     referencia       estado
   ----------------------------------------------------------
   A          0                1                ❌ (error 1.0e+00)
   B          0                -1               ❌ (error 1.0e+00)
   C          0                -5               ❌ (error 1.0e+00)

   Sugerencias: arrastra TODOS los decimales del paso anterior y no
   redondees a la mitad del cálculo.
   En la deflación, el cociente conserva el coeficiente líder del
   polinomio original (y el último valor que obtienes es el residuo).


In [ ]:
pregunta(6)      # los coeficientes de Q(x)

#### Pregunta 5: Coeficientes de Q(x): Deflación polinomial

**Valor:** $\frac{25}{100}$ puntos

**Ejercicio:** Perfil de leva: deflación con una raíz exacta

Con el polinomio **de tu ejercicio de deflación** (la actividad a mano de
esta sección) y la raíz exacta $x = r$ que te da el enunciado, aplica **una
división sintética** para dividir $P(x)$ entre $(x - r)$ y reporta los
**tres coeficientes del polinomio reducido**

$$Q(x) = A\,x^{2} + B\,x + C$$

**en orden, de mayor a menor grado**: primero $A$ (el que acompaña a $x^2$),
luego $B$ (el de $x$) y al final $C$ (el término constante).

Escribe un **número en cada casilla**, sin unidades y sin la $x$. Recuerda
que el cociente **conserva el coeficiente líder** del polinomio original: si
tu polinomio empieza con un coeficiente distinto de 1, el de $Q$ también.

Puedes comprobar tu resultado multiplicando de vuelta: $P(x)$ debe ser igual
a $(x-r)\,Q(x)$.

Escribe **un número en cada casilla**, en el orden del enunciado:

FloatText(value=0.0, description='A =', layout=Layout(width='230px'))

FloatText(value=0.0, description='B =', layout=Layout(width='230px'))

FloatText(value=0.0, description='C =', layout=Layout(width='230px'))

---
### 🧪 Laboratorio · deflación progresiva hasta agotar las raíces

Tu ejercicio tiene una raíz exacta conocida, así que se puede deflactar **en
serie**: encuentra las raíces del polinomio reducido y ya tienes todas.

Aquí se comparan dos caminos y se comprueba el residuo. Fíjate en la última
parte: cuando la raíz **no** es exacta, el residuo es justo el error.

In [ ]:
#@title 🧪 Laboratorio · deflación progresiva y comparación con numpy.roots
a = [float(c) for c in EJ_DF['a']]
r = float(EJ_DF['x0'])

print("Polinomio a = %s   con la raíz conocida r = %g\n" % (a, r))

# --- 1) Deflactar con la raíz conocida -------------------------------
Q, residuo = deflactar(a, r)
print("Deflación entre (x - %g):" % r)
print("   Q(x)    = %s" % Q)
print("   residuo = %g   %s" % (residuo, "(exacta ✅)" if abs(residuo) < 1e-12
                                 else "(NO era raíz exacta ❌)"))

# --- 2) Todas las raíces, por los dos caminos ------------------------
print("\nRaíces del polinomio reducido Q (con numpy.roots):")
raices_q = raices_de(Q)
for z in sorted(raices_q, key=lambda z: (round(z.real, 6), round(z.imag, 6))):
    print("   %s" % ("%.8f" % z.real if abs(z.imag) < 1e-12 else "%.8f %+.8fj"
                     % (z.real, z.imag)))

print("\nTodas las raíces del polinomio original (numpy.roots directo):")
for z in sorted(raices_de(a), key=lambda z: (round(z.real, 6), round(z.imag, 6))):
    print("   %s" % ("%.8f" % z.real if abs(z.imag) < 1e-12 else "%.8f %+.8fj"
                     % (z.real, z.imag)))
print("\n   (deben coincidir: r más las de Q)")

# --- 3) Comprobación: P(x) == (x - r)·Q(x) ---------------------------
def evaluar(coefs, x):
    n = len(coefs) - 1
    return np.sum([c * x ** (n - i) for i, c in enumerate(coefs)])

peor = 0.0
for xt in (-2.0, -0.5, 0.0, 1.7, 4.3):
    izq = evaluar(a, xt)
    der = (xt - r) * evaluar(Q, xt) + residuo
    peor = np.maximum(peor, abs(izq - der) / np.maximum(1.0, abs(izq)))
print("\nComprobación P(x) = (x-r)Q(x) + residuo en 5 puntos: peor diferencia %.2e" % peor)

# --- 4) ¿Y si la raíz NO fuera exacta? -------------------------------
r_mal = r + 0.05
Q_mal, res_mal = deflactar(a, r_mal)
print("\nSi en lugar de r = %g usas r = %g (0.05 de error):" % (r, r_mal))
print("   residuo = %+.6f   <- esto vale P(%.2f), el error de suponerla raíz"
      % (res_mal, r_mal))
print("   P(%.2f) = %+.6f   (comprueba que coincide)" % (r_mal, evaluar(a, r_mal)))

Polinomio a = [1.0, -3.0, -3.0, 10.0]   con la raíz conocida r = 2

Deflación entre (x - 2):
   Q(x)    = [1.0, -1.0, -5.0]
   residuo = 0   (exacta ✅)

Raíces del polinomio reducido Q (con numpy.roots):
   -1.79128785
   2.79128785

Todas las raíces del polinomio original (numpy.roots directo):
   -1.79128785
   2.00000000
   2.79128785

   (deben coincidir: r más las de Q)

Comprobación P(x) = (x-r)Q(x) + residuo en 5 puntos: peor diferencia 1.17e-15

Si en lugar de r = 2 usas r = 2.05 (0.05 de error):
   residuo = -0.142375   <- esto vale P(2.05), el error de suponerla raíz
   P(2.05) = -0.142375   (comprueba que coincide)


---
# 5 · Conclusiones

## Lo que hay que llevarse

1. **La forma anidada baja el costo de $O(n^2)$ a $O(n)$** porque cada paso
   reutiliza el resultado del anterior.
2. **Una sola corrida da tres cosas**: el valor, el polinomio deflactado y —con
   una segunda corrida— la derivada. Es el «dos por uno» que hace que Newton y
   Bairstow evalúen derivadas gratis.
3. **Deflactar es dividir entre $(x-r)$**: baja el grado y conserva las raíces
   que faltan. Si el residuo no es cero, $r$ no era raíz exacta y ese residuo
   mide el error.
4. **El coeficiente líder se conserva** al deflactar. Es el error más común a
   mano.
5. **El costo baja siempre** ($O(n)$ en lugar de $O(n^2)$), que es exactamente lo
   que necesita un microcontrolador evaluando una calibración. El **error** es
   otra historia: Horner suele salir igual o algo mejor, pero quien decide
   cuántos dígitos se pierden es la **condición** de la evaluación ($\kappa$), no
   el método. Hay que medirlo, no suponerlo.

## Autoevaluación rápida

* ¿Por qué $n$ multiplicaciones y no $\frac{n(n+1)}{2}$?
* ¿De dónde sale el $Q(x)$ de la primera corrida, si nunca dividiste nada?
* ¿Por qué $P'(x_0)$ es el **penúltimo** valor de la segunda corrida y no el
  último?
* Si al deflactar el residuo te da $-0.37$, ¿qué significa ese número?
* ¿Por qué no basta con evaluar el polinomio término a término si el resultado
  «sale bien»?

---

### ✏️ Pregunta 7


In [ ]:
#@title 🧪 Pregunta 7 · decisión de ingeniería
pregunta(7)

#### Pregunta 6: Error de redondeo

**Valor:** $\frac{10}{100}$ puntos

Además del tiempo de cómputo, Horner tiene otra ventaja numérica. **¿Cuál?**

a) Obliga a redondear cada paso con 2 decimales

b) Evita calcular potencias grandes como $x^{8}$: no amplifica el error de redondeo

c) Usa números enteros en lugar de flotantes

d) Es más preciso porque ordena los coeficientes de menor a mayor

RadioButtons(layout=Layout(width='170px'), options=(('a)', 0), ('b)', 1), ('c)', 2), ('d)', 3)), value=None)

---
# 6 · Entrega

## Antes de enviar

1. **Revisa tus 7 preguntas.** Si alguna está en blanco el sistema la cuenta
   como error.
2. **Completa la función `horner`.** Se prueba con polinomios que nunca ves.
3. **Imprime la hoja de trabajo manual** (celda de abajo). Va junto con tu
   cuaderno: ahí aparecen tus dos tablas a mano con los valores que reportaste y
   los de referencia, para que tu profesor califique rápido las 2 rúbricas (30 %
   de la calificación).
4. Necesitas **$\ge 90\%$** de los 100 puntos automáticos para poder enviar. Si
   no llegas, corrige y vuelve a ejecutar.

| Sección | Actividad a mano | ¿La reportaste? |
|---|---|---|
| 3 | Horner: $P(x_0)$ y $P'(x_0)$ | `MIS_HR` |
| 4 | Deflación: coeficientes de $Q(x)$ | `MIS_DF` |

---

## Las tres celdas del final

| Celda | Qué hace | Cuántas veces |
|---|---|---|
| 📝 **Calificar** | revisa tus respuestas y te dice cuántos puntos llevas y **cuánto suma eso a la nota final** ($0.70\times$ automático). **No envía nada.** | las que quieras |
| 📤 **Enviar** | manda tu calificación a la hoja del profesor. | **solo 2 intentos** |
| 🔎 **Comprobar** | lee lo que la hoja ya tiene guardado de ti: intentos usados, último total y la nota final que llevas. **No envía nada.** | las que quieras |

La **nota final** es $0.70\,(\text{automático})+0.30\,(\text{manual})$, así que las celdas te muestran solo lo que aporta el automático: el 30 % manual lo pone tu profesor con la hoja de trabajo impresa.

**Calificar no gasta intentos.** Úsala todas las veces que necesites: corrige, vuelve a calificar y solo cuando estés conforme envía. Si el envío se rechaza por no llegar al 90 %, tampoco se gasta: en la hoja no se guarda nada.

> Si te quedó alguna pregunta mal, **no** borres tu trabajo: primero entiende
> **por qué** falló. Los errores de esta tarea son exactamente los que aparecen
> en el examen final (y en la siguiente entrega, donde Müller y Bairstow usan
> estas mismas divisiones sintéticas).


In [35]:
#@title 📋 Hoja de trabajo manual (entrégala con tu cuaderno)
hoja_manual()

HOJA DE TRABAJO MANUAL · U2_T3A

Forma anidada de Horner · Curva de un motor de CD (grado 3)
  polinomio  : [1, -4, 7, 1]
  punto      : 2.3
  referencia : P(x_0) = 8.107, P'(x_0) = 4.47
  alumno     : 0, 0
  ecuación   : NO
  tabla      : revisar

Deflación polinomial · Perfil de leva: deflación con una raíz exacta
  polinomio  : [1, -3, -3, 10]
  punto      : 2
  referencia : A = 1, B = -1, C = -5
  alumno     : 0, 0, 0
  ecuación   : NO
  tabla      : revisar

Califica cada rúbrica de 0 a 10 en la hoja U2_T3A_Manual.
Nota final = 0.70*automático + 0.30*manual


In [36]:
#@title 📝 Calificar (sin enviar) — úsala las veces que quieras
# Revisa tus respuestas y te dice cuántos puntos llevas.
# NO envía nada: puedes corregir y volver a calificar sin gastar intentos.
puntos, maximo = calificar()

#    estado       puntos      respuesta
--------------------------------------------------------------
1    ⚠️           0/10        None
2    ❌            0/25        programa (3 casos)
3    ❌            0/15        0.0
4    ❌            0/15        0.0
5    ❌            0/25        [0, 0, 0]
6    ⚠️           0/10        None
--------------------------------------------------------------
AUTOMÁTICO: 0.0 / 100  =  0.0 %
NOTA FINAL (ponderada): 0.0 / 100  =  0.70 × 0.0 %  +  0.30 × manual
   (falta el 30 % manual: las 2 rúbricas de la hoja de trabajo)


In [37]:
#@title 📤 Enviar la calificación (máximo 2 intentos)
# Ojo: solo tienes DOS envíos y se cuentan en la hoja del profesor.
# Si todavía no estás conforme, primero usa la celda de calificar de arriba.
enviar(alumno_id)

#    estado       puntos      respuesta
--------------------------------------------------------------
1    ⚠️           0/10        None
2    ❌            0/25        programa (3 casos)
3    ❌            0/15        0.0
4    ❌            0/15        0.0
5    ❌            0/25        [0, 0, 0]
6    ⚠️           0/10        None
--------------------------------------------------------------
AUTOMÁTICO: 0.0 / 100  =  0.0 %
NOTA FINAL (ponderada): 0.0 / 100  =  0.70 × 0.0 %  +  0.30 × manual
   (falta el 30 % manual: las 2 rúbricas de la hoja de trabajo)
⛔ Aún no puedes enviar: necesitas al menos 90 % (90 puntos de 100). Corrige y vuelve a intentarlo.
   (este intento NO se gastó: no se guardó nada en la hoja)


In [38]:
#@title 🔎 Comprobar lo que ya está guardado (no envía nada)
# Lee la hoja del profesor: cuántos intentos llevas y tu último total.
# Es de solo lectura, así que puedes ejecutarla cuantas veces quieras.
consultar_calificacion()

NC 16330887  ·  U2_T3A
   Intentos usados : 0 de 2
   Te quedan       : 2
   Último total    : (todavía sin nota)
   Estado          : en progreso
   Todavía tienes 2 intentos: revisa con calificar() y luego envía.
